In [1]:
!apt update -q && apt-get install -qy openjdk-21-jdk-headless > /dev/null 2>&1

import subprocess
from IPython.core.magic import register_cell_magic

class PersistentJShell:
    def __init__(self):
        self.proc = subprocess.Popen(
            ["jshell", "-s"],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

    def execute(self, code):
        if self.proc.poll() is not None:
            raise RuntimeError("JShell process has terminated.")

        marker = "___JSHELL_CELL_END___"
        payload = code.strip() + f'\nSystem.out.println("{marker}");\n'
        self.proc.stdin.write(payload)
        self.proc.stdin.flush()

        lines = []
        while True:
            line = self.proc.stdout.readline()
            if not line:
                break
            if marker in line:
                break
            lines.append(line)

        return "".join(lines)

_jshell_instance = PersistentJShell()

@register_cell_magic
def jshell(line, cell):
    out = _jshell_instance.execute(cell)
    if out:
        print(out, end="")


Get:1 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Get:5 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [73.2 kB]
Get:9 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packages [1,905 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,028 kB]
Get:11 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1,549 kB]
Get:12 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1,300 kB]
Get:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InR

In [2]:
%%jshell

void runPy(String code) throws Exception {
    String marker = "___END_OF_PYTHON_CELL___";

    // Base64 encode code payload to bypass all quote & newline syntax errors
    String b64 = java.util.Base64.getEncoder().encodeToString(code.getBytes(java.nio.charset.StandardCharsets.UTF_8));

    // Guarantee marker printing using try...finally and display trailing expressions using ast
    String payload = String.format(
        "import base64, ast, traceback\n" +
        "try:\n" +
        "    code_str = base64.b64decode('%s').decode('utf-8')\n" +
        "    tree = ast.parse(code_str)\n" +
        "    if tree.body and isinstance(tree.body[-1], ast.Expr):\n" +
        "        exec(compile(ast.Module(body=tree.body[:-1], type_ignores=[]), '<string>', 'exec'), globals())\n" +
        "        res = eval(compile(ast.Expression(body=tree.body[-1].value), '<string>', 'eval'), globals())\n" +
        "        if res is not None:\n" +
        "            print(res)\n" +
        "    else:\n" +
        "        exec(compile(tree, '<string>', 'exec'), globals())\n" +
        "except Exception:\n" +
        "    traceback.print_exc()\n" +
        "finally:\n" +
        "    print('%s', flush=True)\n",
        b64, marker
    );

    pyIn.println(payload);

    String line;
    while ((line = pyOut.readLine()) != null) {
        if (line.contains(marker)) break;
        System.out.println(line);
    }
}

import java.io.BufferedReader;
import java.io.InputStreamReader;
import java.io.PrintStream;
import java.io.IOException;

Process pythonProcess;
PrintStream pyIn;
BufferedReader pyOut;

try {
    ProcessBuilder pb = new ProcessBuilder("python3", "-i", "-u");
    // Redirect stderr to stdout to prevent pipe deadlock and capture Python errors
    pb.redirectErrorStream(true);
    pythonProcess = pb.start();

    pyIn = new PrintStream(pythonProcess.getOutputStream(), true);
    pyOut = new BufferedReader(new InputStreamReader(pythonProcess.getInputStream()));

    System.out.println("Python interpreter initialized safely.");
} catch (IOException e) {
    e.printStackTrace();
}

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Sep 28, 2026 3:12:26 PM java.util.prefs.FileSystemPreferences$1 run
INFO: Created user preferences directory.
->  void runPy(String code) throws Exception {
>>         String marker =  "___END_OF_PYTHON_CELL___";
>>     
>>         // Base64 encode code payload to bypass all quote & newline syntax errors
>>          String b64 = java.util.Base64.getEncoder().encodeToString(code.getBytes(java.nio.charset.StandardCharsets.UTF_8));
>>      
>>         // Guarantee marker printing using try...finally and display trailing expressions using ast
>>         String payload = String.format(
>>              "import base64, ast, traceback\n" +
>>             "try:\n" +
>>   

In [7]:
%%jshell

runPy("""
import os

# 1. Clone the repository
os.system('git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming')

# 2. Change the working directory permanently for this Python session
os.chdir('/content/BITS_programming/assignments/assignment_8')

# 3. Now all relative paths work correctly
os.system('pip install -r requirements.txt')
os.system('python scripts/setup_baseline.py')
os.system('python scripts/m8_l1_quantize_onnx.py')
os.system('python scripts/m8_l2_benchmark.py')
""");

->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... fatal: destination path '/content/BITS_programming' already exists and is not an empty directory.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 19.9 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.2 which 

In [8]:
%%jshell

runPy("""
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Current Time in IST: 2026-09-28 20:53:24
